![](https://github.com/ibmm-unibe-ch/FrankenMSA/blob/dev/app/assets/frankenmsa_header.png?raw=true)

# FrankenMSA-Colab
This notebook launches the [FrankenMSA App](https://github.com/ibmm-unibe-ch/FrankenMSA/tree/main/) **in Google Colab** to provide a GUI for manipulating Multiple Sequence Alignments (MSAs).


Tip: use “Runtime” → “Run all” (or `Ctrl + F9`) to execute all cells.

In [ ]:
FRANKEN_GIT_BRANCH = "main" # change as needed (default: main)
FRANKEN_GIT_URL = "https://github.com/ibmm-unibe-ch/FrankenMSA.git"

In [ ]:
#@title Install Prerequisites

import os
os.system("pip install termcolor gitpython ipywidgets ipython dotenv > /dev/null 2>&1")

import git, sys, importlib
from pathlib import Path
from termcolor import colored
import dotenv

dotenv.load_dotenv()
os.environ["ON_COLAB"] = "1"




def warn(msg):
    print(colored("[WARNING] ", "yellow") + msg, file=sys.stderr)

def info(msg):
    print(colored("[INFO] ", "cyan") + msg)

info("Installed prerequisite packages.")

In [ ]:
#@title Forwarding Options

import ipywidgets as widgets
from IPython.display import display

_forwarding_dropdown = widgets.Dropdown(
    options=[
        ("Google Native (Colab only)", "native"),
        ("ngrok (shareable link)", "ngrok"),
    ],
    value="native",
    description="Forwarding:",
)

_port_input = widgets.IntText(
    value=8050,
    description="Port:",
    min=1024,
    max=65535,
)

_status = widgets.Output()

use_ngrok = lambda: _forwarding_dropdown.value == "ngrok"

def _on_forwarding_change(change):
    if change["name"] != "value":
        return
    with _status:
        _status.clear_output()
        if change["new"] == "ngrok":
            info("ngrok forwarding will create a shareable public link (requires auth token)")
        else:
            info("Google Native forwarding uses Colab's built-in port forwarding")

_forwarding_dropdown.observe(_on_forwarding_change, names="value")
display(widgets.VBox([_forwarding_dropdown, _port_input, _status]))


In [ ]:
#@title Install FrankenMSA

FORCE_REINSTALL = False #@param {type:"boolean"}

if Path("frankenmsa").exists() and not FORCE_REINSTALL:
    warn("FrankenMSA directory already exists; skipping clone.")
else:
    os.system("rm -rf FrankenMSA frankenmsa app; rm -f *.py")
    info(f"Cloning FrankenMSA from {FRANKEN_GIT_URL} (branch: {FRANKEN_GIT_BRANCH})...")
    git.Repo.clone_from(FRANKEN_GIT_URL, "FrankenMSA", branch=FRANKEN_GIT_BRANCH)
    info("FrankenMSA cloned.")

    os.system("mv FrankenMSA/app .; mv FrankenMSA/frankenmsa .; mv FrankenMSA/setup.py .; rm -rf FrankenMSA; pip install -e .")
    info("FrankenMSA installed.")

# kill any existing instances
!pkill -f "app/app.py" 2>/dev/null || true
!pkill -f "gunicorn" 2>/dev/null || true
!pkill -f "ngrok" 2>/dev/null || true

In [ ]:
#@title Launch FrankenMSA App (without ngrok sharing)
if not use_ngrok():
    from google.colab import output
    import subprocess, time, threading

    PORT = _port_input.value

    # setup environment variables for subprocess app launch
    env = os.environ.copy()
    env["PORT"], env["HOST"] = str(PORT), "0.0.0.0"
    env["PYTHONPATH"] = "/content:" + env.get("PYTHONPATH", "")
    env["FRANKEN_COLAB"] = "1"
    env["ON_COLAB"] = "1"
    env["FRANKEN_RENDER_MODE"] = "inline"  # Use inline mode for Colab native forwarding
    env["FRANKEN_USE_NGROK"] = "0"

    # Start the app as a subprocess
    proc = subprocess.Popen(
        [sys.executable, "app/app.py"],
        cwd=str(Path.cwd().resolve()),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    # Wait for app to start
    start = time.time()
    lines = []
    app_started = False
    while time.time() - start < 25:
        ln = proc.stdout.readline()
        if ln:
            lines.append(ln.rstrip())
            if "Dash is running" in ln or "Running on" in ln:
                app_started = True
                break
        else:
            time.sleep(0.2)

    if app_started:
        # Use Colab's native port forwarding
        info(f"🌐 Opening FrankenMSA app on port {PORT} with Colab native forwarding...")
        output.serve_kernel_port_as_window(PORT)
        info("✅ A link to the app should now be displayed above")
    else:
        warn("App may not have started properly. Check logs below.")
        print("\n---- recent logs ----")
        print("\n".join(lines))
        print("---------------------")


    # Tail logs in background
    def tail_logs():
        while True:
            line = proc.stdout.readline()
            if not line:
                time.sleep(0.2)
                continue
            print(line, end="")

    log_thread = threading.Thread(target=tail_logs, daemon=True)
    log_thread.start()

    info("📡 App logs are being tailed in the background")


In [ ]:
#@title Launch FrankenMSA App (with ngrok sharing)
if use_ngrok():

    try:
        import pyngrok
    except:
        info("Installing pyngrok...")
        os.system("pip install pyngrok > /dev/null 2>&1")
        info("pyngrok installed.")
    try:
        from pyngrok import ngrok

    except:
        raise ImportError("Pyngrok could not be installed")

    from pyngrok import ngrok, conf
    import getpass, re

    token = os.environ.get("NGROK_AUTH_TOKEN", "").strip()
    if not token:
        warn("No ngrok auth token found in NGROK_AUTH_TOKEN env variable.")
        warn("You can sign up for a free ngrok account at https://ngrok.com/")
        warn("To avoid entering the token every time, the token is set it in the NGROK_AUTH_TOKEN environment variable after entering.")

        token = getpass.getpass("Enter ngrok authtoken (hidden): ").strip().strip("'").strip('"')
        os.environ["NGROK_AUTH_TOKEN"] = token

        info("ngrok auth token set as environment variable.")
    conf.get_default().auth_token = token

    public_url = None
    PORT = _port_input.value
    try:
        for t in ngrok.get_tunnels():
            addr = (t.config or {}).get("addr", "")
            if addr.endswith(f":{PORT}"):
                public_url = t.public_url
                print("♻️ Reusing existing tunnel:", public_url)
                break

        if not public_url:
            tun = ngrok.connect(addr=f"0.0.0.0:{PORT}", proto="http")
            public_url = tun.public_url
            print("✅ Created new tunnel:", public_url)

    except Exception as e:
        msg = str(e)
        m = re.search(r"https?://[a-z0-9\-]+\.ngrok-[\w\-]+\.(?:dev|app)", msg)
        if m:
            public_url = m.group(0)
            warn("♻️ Using tunnel from error message:", public_url)
        else:
            raise e

    import subprocess, time

    # setup environment variables for subprocess app launch
    env = os.environ.copy()
    env["PORT"], env["HOST"] = str(PORT), os.environ.get("HOST", "0.0.0.0")
    env["PYTHONPATH"] = "/content:" + env.get("PYTHONPATH", "")
    env["FRANKEN_COLAB"] = "1"
    env["ON_COLAB"] = "1"
    env["FRANKEN_RENDER_MODE"] = "external"
    env["FRANKEN_USE_NGROK"] = "1"
    if public_url:
        env["COLAB_TUNNEL_URL"] = public_url
    else:
        env.pop("COLAB_TUNNEL_URL", None)

    proc = subprocess.Popen(
        [sys.executable, "app/app.py"],
        cwd=str(Path.cwd().resolve()),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    start = time.time()
    lines = []
    while time.time() - start < 25:
        ln = proc.stdout.readline()
        if ln:
            lines.append(ln.rstrip())
            if "Running on" in ln or "Dash is running" in ln:
                break
        else:
            time.sleep(0.2)

    display_host = env["HOST"] if env["HOST"] not in {"0.0.0.0", "::"} else "0.0.0.0"
    open_target = public_url if public_url else f"http://{display_host}:{PORT}"

    print("\n---- recent logs ----")
    print("\n".join(lines[-20:]))
    print("---------------------")
    info(f"🌐 Open: {open_target}")
    # print("✅ Look for the banner 'IF-build:xxxx' on the page header to confirm version")
    print("📡 Tailing FrankenMSA app logs (Ctrl+C to stop):")
    while True:
        line = proc.stdout.readline()
        if not line:
            time.sleep(0.2)
            continue
        print(line, end="")